# APR Amazon 지수 — 조회 · 진단 · 시각화 V2

`apr_us_amazon_keepa_query_v2.ipynb`

---

수집 노트북(`apr_us_amazon_keepa_v2`)이 적재한 DB를 읽어 **진단하고 시각화**한다.
수집과 조회를 분리하는 것은 토큰을 쓰지 않고 몇 번이든 다시 볼 수 있게 하기 위함이다.

## 이 노트북이 먼저 답해야 하는 질문

> 지수 하락이 **수요 감소**인가, **데이터 커버리지 붕괴**인가?

`gmv_index_all`은 그 시점 활성 ASIN의 **합계**다. 따라서 ASIN이 패널에서 빠지면
수요와 무관하게 지수가 떨어진다. Keepa는 ASIN마다 마지막 갱신 시점이 다르고,
수집 노트북의 forward-fill은 `FFILL_LIMIT_D`(기본 7일)에서 끊긴다.
결과적으로 **수집일 직전 구간은 거의 항상 커버리지가 무너진다.**

활성 ASIN 수가 22→13으로 41% 줄었는데 지수가 -36% 라면,
그 하락의 대부분은 수요가 아니라 산술이다.

## 대응 — 3종 지수 비교

| 지수 | 정의 | 커버리지 민감도 |
|---|---|---|
| `gmv_sum` | 활성 ASIN 합계 | **높음** (원본) |
| `gmv_mean` | 활성 ASIN 평균 | 낮음 |
| `gmv_balanced` | 균형 패널 합계 | 없음 |

`gmv_balanced`가 가장 신뢰할 만하다. 비교 구간 양 끝에서 모두 관측되는
ASIN만 남긴 것이라, 구성 변화가 지수를 흔들지 않는다.
셋이 같은 방향이면 진짜 신호, 갈라지면 커버리지 문제다.

---
## V2 변경사항

| 항목 | 내용 |
|---|---|
| DB 연결 | `config.get_engine(config.get_db_info())` 규약으로 교체 |
| `sys.path` | `config.py` 의 `DATA.*` import 를 위해 프로젝트 루트도 추가 |

---
**V2** | 2026-08

## 1. 사용자 입력 변수

In [ ]:
# ============================================================================
# USER INPUT
# ============================================================================

TICKER          = "278470"      # 조회 대상
DOMAIN          = "US"
COLLECTED_AT    = None          # 특정 vintage 조회. None이면 최신

# ---------- 커버리지 진단 ----------
COVERAGE_MIN    = 0.70          # 최대 ASIN 수 대비 이 비율 미만인 날은 신뢰 구간에서 제외
TRIM_TAIL       = True          # 꼬리 구간(커버리지 붕괴) 자동 절단
TRIM_HEAD       = True          # 초기 구간(SKU 부족) 자동 절단

# ---------- 지수 재계산 ----------
ALPHA           = 0.70          # 수집 노트북과 동일하게
RECOMPUTE       = True          # True면 일별 원본에서 3종 지수를 다시 계산
BALANCED_MIN_COVER = 0.90       # 균형 패널 편입 기준 (분석기간 중 관측 비율)

# ---------- 비교 구간 ----------
WINDOW_D        = 90            # 국면 비교 창 (일)
REBASE_DATE     = None          # 지수 100 기준일. None이면 분석 시작일

# ---------- 시각화 ----------
SHOW_PLOTS      = True
TOP_N_SKU       = 12            # 개별 SKU 차트에 표시할 상위 개수
FIG_W           = 14
OVERLAY_TRENDS  = True          # Google Trends 지표 겹쳐보기

# ---------- 테이블명 ----------
TBL_ASIN_MASTER = "amazon_asin_master"
TBL_BSR_DAILY   = "amazon_bsr_daily"
TBL_BRAND_INDEX = "amazon_brand_index"
TBL_TRENDS_MET  = "apr_us_trends_metrics"
TBL_APR_ACTUAL  = "apr_us_quarterly_actual"


## 2. 환경 · DB 연결

In [ ]:
import os
import sys
import json
import warnings
from datetime import date, datetime
from typing import Optional, Union, List, Dict, Any, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

# ============================================================================
# 경로 자동탐지 — 데스크톱 / 노트북 어디서 실행해도 DATA/ 를 찾아낸다
# ============================================================================
# 탐색 순서:
#   1) BASE_PATH_OVERRIDE (수동 지정)
#   2) 환경변수 STOCK_BASE_PATH
#   3) 현재 작업 디렉토리에서 상위로 6단계
#   4) 알려진 고정 경로
#   5) 홈 디렉토리 기반 얕은 탐색 (OneDrive / 바탕 화면 / Desktop 등)
# ----------------------------------------------------------------------------

BASE_PATH_OVERRIDE = None   # 전부 실패하면 여기에 경로를 직접 문자열로 넣으십시오

KNOWN_PATHS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

PROJECT_DIR_NAMES = ["stock_forecast", "investment_strategy", "investment"]


def _has_data(p: Optional[str]) -> bool:
    if not p:
        return False
    try:
        return os.path.isdir(os.path.join(p, "DATA"))
    except Exception:
        return False


def resolve_base_path() -> Tuple[Optional[str], List[Tuple[str, str]]]:
    """DATA/ 를 포함한 프로젝트 루트를 탐색. (경로, 시도이력) 반환."""
    tried: List[Tuple[str, str]] = []

    # (1) 수동 지정
    if BASE_PATH_OVERRIDE:
        tried.append(("override", BASE_PATH_OVERRIDE))
        if _has_data(BASE_PATH_OVERRIDE):
            return BASE_PATH_OVERRIDE, tried

    # (2) 환경변수
    envp = os.environ.get("STOCK_BASE_PATH")
    if envp:
        tried.append(("env", envp))
        if _has_data(envp):
            return envp, tried

    # (3) cwd에서 상위로 — 노트북이 프로젝트 하위 어디에 있어도 잡힌다
    cur = os.path.abspath(os.getcwd())
    for _ in range(6):
        tried.append(("cwd-up", cur))
        if _has_data(cur):
            return cur, tried
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent

    # (4) 알려진 고정 경로
    for p in KNOWN_PATHS:
        tried.append(("known", p))
        if _has_data(p):
            return p, tried

    # (5) 홈 기반 얕은 탐색 — OneDrive / 한글·영문 바탕화면 변형 모두 대응
    home = os.path.expanduser("~")
    roots = [
        home,
        os.path.join(home, "OneDrive"),
        os.path.join(home, "Desktop"),
        os.path.join(home, "바탕 화면"),
        os.path.join(home, "OneDrive", "Desktop"),
        os.path.join(home, "OneDrive", "바탕 화면"),
        os.path.join(home, "Documents"),
        os.path.join(home, "PyCharmMiscProject"),
    ]
    for root in roots:
        if not os.path.isdir(root):
            continue
        tried.append(("home", root))
        if _has_data(root):
            return root, tried
        for nm in PROJECT_DIR_NAMES:
            cand = os.path.join(root, nm)
            if _has_data(cand):
                return cand, tried
        # 한 단계 더 들어가서 확인
        try:
            for sub in sorted(os.listdir(root)):
                subp = os.path.join(root, sub)
                if not os.path.isdir(subp):
                    continue
                if _has_data(subp):
                    return subp, tried
                for nm in PROJECT_DIR_NAMES:
                    cand = os.path.join(subp, nm)
                    if _has_data(cand):
                        return cand, tried
        except (PermissionError, OSError):
            continue

    return None, tried


BASE_PATH, _tried = resolve_base_path()

if BASE_PATH is None:
    print("=" * 78)
    print("[ERROR] DATA/ 폴더를 포함한 프로젝트 루트를 찾지 못했습니다.")
    print("=" * 78)
    print("\n시도한 경로 (%d개):" % len(_tried))
    for how, p in _tried[:20]:
        print("  [%-8s] %s" % (how, p))
    print("\n해결 방법 — 아래 중 하나:")
    print("  (A) 이 셀 상단의 BASE_PATH_OVERRIDE 에 경로를 직접 입력")
    print(r"      예: BASE_PATH_OVERRIDE = r'D:\work\investment_strategy'")
    print("  (B) 환경변수 STOCK_BASE_PATH 설정 후 Jupyter 재시작")
    print('      PowerShell:  setx STOCK_BASE_PATH "D:\\work\\investment_strategy"')
    print("  (C) KNOWN_PATHS 리스트에 이 PC의 경로를 추가")
    print("\n현재 작업 디렉토리:", os.getcwd())
    BASE_PATH = os.getcwd()
    DATA_PATH = os.path.join(BASE_PATH, "DATA")
else:
    DATA_PATH = os.path.join(BASE_PATH, "DATA")
    print("BASE_PATH :", BASE_PATH)
    print("DATA_PATH :", DATA_PATH)

if DATA_PATH not in sys.path:
    sys.path.append(DATA_PATH)
# config.py 가 `from DATA.stock_invest_function import ...` 형태로 import 하므로
# DATA/ 뿐 아니라 그 부모(프로젝트 루트)도 sys.path 에 있어야 한다.
if BASE_PATH not in sys.path:
    sys.path.insert(0, BASE_PATH)

print("python    :", sys.version.split()[0])
print("pandas    :", pd.__version__)


In [ ]:
from sqlalchemy import text

ENGINE = None
DB_AVAILABLE = False
DB_INFO = {}


def make_engine():
    """프로젝트의 DB 접속 규약에 맞춰 엔진을 만든다.

    이 프로젝트는 config.py 가 get_db_info() / get_engine(db_info) 를 제공한다.
    (get_engine 은 stock_invest_function 이 아니라 config 에 있고 인자를 받는다)
    다른 환경에서도 죽지 않도록 여러 경로를 순차 시도한다.
    """
    errs = []

    # (1) config.get_engine(config.get_db_info())  ← 이 프로젝트의 표준
    try:
        import config
        info = config.get_db_info()
        return config.get_engine(info), "config.get_engine(db_info)", info
    except Exception as e:
        errs.append("config(db_info): %s" % repr(e)[:110])

    # (2) config.get_engine() 무인자 변형
    try:
        import config
        return config.get_engine(), "config.get_engine()", {}
    except Exception as e:
        errs.append("config(): %s" % repr(e)[:110])

    # (3) stock_invest_function 에 get_engine 이 있는 경우
    try:
        from stock_invest_function import get_engine as _ge
        try:
            return _ge(), "stock_invest_function.get_engine()", {}
        except TypeError:
            import config
            info = config.get_db_info()
            return _ge(info), "stock_invest_function.get_engine(db_info)", info
    except Exception as e:
        errs.append("sif: %s" % repr(e)[:110])

    raise RuntimeError("모든 DB 연결 경로 실패 -> " + " | ".join(errs))


try:
    ENGINE, _how, DB_INFO = make_engine()
    with ENGINE.connect() as _c:
        _dbname = _c.execute(text("SELECT DATABASE()")).scalar()
    DB_AVAILABLE = True
    print("[DB] 연결 성공 - %s / database=%s" % (_how, _dbname))
except Exception as e:
    print("[DB] 연결 실패:", repr(e)[:280])
    print()
    print("  확인 사항:")
    print("   1) config.py 가 프로젝트 루트(DATA/ 의 부모)에 있는지")
    print("      현재 BASE_PATH: %s" % BASE_PATH)
    print("      config.py 존재: %s" % os.path.exists(os.path.join(BASE_PATH, "config.py")))
    if DB_INFO:
        print("   2) MySQL 서버 기동 여부 - %s:%s / db=%s"
              % (DB_INFO.get("host"), DB_INFO.get("port"), DB_INFO.get("database")))
    else:
        print("   2) MySQL 서버가 기동 중인지, 포트가 열려 있는지")
    print("   3) 수동 테스트:")
    print("      import config; e = config.get_engine(config.get_db_info())")
    print("      from sqlalchemy import text")
    print("      print(e.connect().execute(text('SELECT 1')).scalar())")


## 3. 데이터 로드

In [ ]:
daily = pd.DataFrame()
master = pd.DataFrame()
idx_db = pd.DataFrame()

if DB_AVAILABLE:
    q_master = "SELECT * FROM {t} WHERE ticker=:tk AND domain=:dm".format(t=TBL_ASIN_MASTER)
    master = pd.read_sql(text(q_master), ENGINE, params={"tk": TICKER, "dm": DOMAIN})

    if len(master):
        asin_list = master["asin"].tolist()
        ph = ",".join([":a%d" % i for i in range(len(asin_list))])
        prm = {"a%d" % i: a for i, a in enumerate(asin_list)}
        prm["dm"] = DOMAIN
        q_daily = ("SELECT * FROM {t} WHERE domain=:dm AND asin IN ({ph})"
                   .format(t=TBL_BSR_DAILY, ph=ph))
        daily = pd.read_sql(text(q_daily), ENGINE, params=prm)
        if len(daily):
            daily["date"] = pd.to_datetime(daily["date"])

    q_idx = "SELECT * FROM {t} WHERE ticker=:tk AND domain=:dm".format(t=TBL_BRAND_INDEX)
    idx_db = pd.read_sql(text(q_idx), ENGINE, params={"tk": TICKER, "dm": DOMAIN})
    if len(idx_db):
        idx_db["date"] = pd.to_datetime(idx_db["date"])
        idx_db["collected_at"] = pd.to_datetime(idx_db["collected_at"])
        # vintage 선택
        if COLLECTED_AT:
            idx_db = idx_db[idx_db["collected_at"] == pd.to_datetime(COLLECTED_AT)]
        else:
            latest = idx_db["collected_at"].max()
            idx_db = idx_db[idx_db["collected_at"] == latest]
            print("[로드] vintage:", latest.date())

print("[로드] master %d행 / daily %d행 / index %d행"
      % (len(master), len(daily), len(idx_db)))
if len(daily):
    print("       기간: %s ~ %s" % (daily["date"].min().date(), daily["date"].max().date()))
    print("       ASIN: %d개" % daily["asin"].nunique())


## 4. 커버리지 진단 — 가장 먼저 볼 것

지수 해석 전에 **패널 구성이 시간에 따라 어떻게 변했는지**를 봐야 한다.
활성 ASIN 수가 흔들리는 구간의 지수 변화는 수요 신호로 읽으면 안 된다.

특히 **수집일 직전 1~2주**는 거의 항상 커버리지가 무너진다.
Keepa가 모든 ASIN을 같은 날 갱신하지 않기 때문이다.

In [ ]:
cov = pd.DataFrame()

if len(daily):
    d = daily.copy()
    # BSR이 실제로 있는 날만 '활성'으로 센다
    d["has_bsr"] = d["bsr_root"].notna().astype(int)
    d["has_obs"] = ((d["bsr_root"].notna()) & (d.get("is_ffilled", 0) == 0)).astype(int)

    cov = pd.DataFrame({
        "n_active":   d.groupby("date")["has_bsr"].sum(),
        "n_observed": d.groupby("date")["has_obs"].sum(),
        "n_rows":     d.groupby("date").size(),
    })
    cov["n_max"] = cov["n_active"].rolling(180, min_periods=1).max()
    cov["cover_ratio"] = cov["n_active"] / cov["n_max"].replace(0, np.nan)

    print("=" * 78)
    print("커버리지 진단")
    print("=" * 78)
    print("  전체 기간 최대 활성 ASIN : %d" % cov["n_active"].max())
    print("  최근 30일 평균 활성 ASIN : %.1f" % cov["n_active"].tail(30).mean())
    print("  최종일 활성 ASIN         : %d" % cov["n_active"].iloc[-1])
    print()

    # 꼬리에서 커버리지가 무너진 지점 탐지
    tail = cov[cov["cover_ratio"] < COVERAGE_MIN]
    if len(tail):
        recent_break = tail.index[tail.index >= cov.index.max() - pd.Timedelta(days=120)]
        if len(recent_break):
            print("  [경고] 최근 120일 내 커버리지 %.0f%% 미만 구간 %d일 발견"
                  % (COVERAGE_MIN * 100, len(recent_break)))
            print("         최초 붕괴일: %s" % recent_break.min().date())
            print()
            print("  이 구간의 지수 하락은 수요가 아니라 패널 축소일 가능성이 높습니다.")
            print("  아래 5장에서 커버리지 중립 지수로 재확인하십시오.")
    print()
    print(cov.tail(15).round(3).to_string())
else:
    print("[커버리지] 데이터 없음")


In [ ]:
# ---------------------------------------------------------------- 분석 구간 확정
ANALYSIS_START, ANALYSIS_END = None, None

if len(cov):
    valid = cov[cov["cover_ratio"] >= COVERAGE_MIN]

    if TRIM_HEAD and len(valid):
        ANALYSIS_START = valid.index.min()
    else:
        ANALYSIS_START = cov.index.min()

    if TRIM_TAIL and len(valid):
        # 마지막 연속 유효구간의 끝
        ANALYSIS_END = valid.index.max()
    else:
        ANALYSIS_END = cov.index.max()

    trimmed_tail = (cov.index.max() - ANALYSIS_END).days
    trimmed_head = (ANALYSIS_START - cov.index.min()).days

    print("=" * 78)
    print("분석 구간 확정")
    print("=" * 78)
    print("  원본 : %s ~ %s" % (cov.index.min().date(), cov.index.max().date()))
    print("  분석 : %s ~ %s" % (ANALYSIS_START.date(), ANALYSIS_END.date()))
    print("  절단 : 앞 %d일 / 뒤 %d일" % (trimmed_head, trimmed_tail))
    if trimmed_tail > 3:
        print()
        print("  ※ 뒤쪽 %d일은 커버리지 부족으로 제외했습니다." % trimmed_tail)
        print("    이 구간을 포함해 최근 추세를 판단하면 하락을 과대평가하게 됩니다.")


## 5. 커버리지 중립 지수 3종

같은 원본에서 세 가지 방식으로 지수를 만들어 비교한다.

- **`gmv_sum`** — 활성 ASIN 합계. 수집 노트북의 `gmv_index_all`과 동일. SKU 확장을 반영하지만 커버리지에 취약
- **`gmv_mean`** — 활성 ASIN 평균. ASIN 수가 변해도 흔들리지 않음
- **`gmv_balanced`** — 분석기간 중 `BALANCED_MIN_COVER` 이상 관측된 ASIN만 합산. 구성 고정

**셋이 같은 방향이면 진짜 신호다.** 갈라지면 커버리지 문제이며,
그때는 `gmv_balanced`를 믿어야 한다.

In [ ]:
idx = pd.DataFrame()

if RECOMPUTE and len(daily) and ANALYSIS_START is not None:
    d = daily[(daily["date"] >= ANALYSIS_START) & (daily["date"] <= ANALYSIS_END)].copy()

    # 품절 마스킹 — 공급 문제이지 수요 신호가 아니다
    if "is_oos" in d.columns:
        d.loc[d["is_oos"] == 1, "bsr_root"] = np.nan

    bsr = pd.to_numeric(d["bsr_root"], errors="coerce")
    bsr = bsr.where(bsr > 0)
    price = (pd.to_numeric(d["price_new"], errors="coerce")
             .fillna(pd.to_numeric(d.get("price_buybox"), errors="coerce"))
             .fillna(pd.to_numeric(d.get("price_amazon"), errors="coerce")))

    d["gmv_proxy"] = (bsr ** (-ALPHA)) * price
    d["neg_log_bsr"] = -np.log(bsr)

    # 균형 패널: 분석기간 중 관측 비율이 기준 이상인 ASIN
    n_days = d["date"].nunique()
    obs_ratio = d[d["gmv_proxy"].notna()].groupby("asin")["date"].nunique() / n_days
    balanced = obs_ratio[obs_ratio >= BALANCED_MIN_COVER].index.tolist()
    print("[균형패널] %d개 ASIN (전체 %d개 중, 기준 %.0f%%)"
          % (len(balanced), d["asin"].nunique(), BALANCED_MIN_COVER * 100))

    dev_map = master.set_index("asin")["is_device"].to_dict() if len(master) else {}
    d["is_device"] = d["asin"].map(dev_map).fillna(0).astype(int)

    g = d.groupby("date")
    idx = pd.DataFrame(index=sorted(d["date"].unique()))
    idx.index.name = "date"
    idx["gmv_sum"]      = g["gmv_proxy"].sum(min_count=1)
    idx["gmv_mean"]     = g["gmv_proxy"].mean()
    idx["gmv_balanced"] = d[d["asin"].isin(balanced)].groupby("date")["gmv_proxy"].sum(min_count=1)
    idx["gmv_device"]   = d[d["is_device"] == 1].groupby("date")["gmv_proxy"].sum(min_count=1)
    idx["n_active"]     = g["gmv_proxy"].count()

    wsum = g["gmv_proxy"].sum(min_count=1)
    num = d.assign(_w=d["gmv_proxy"] * d["neg_log_bsr"]).groupby("date")["_w"].sum(min_count=1)
    idx["brand_strength"] = num / wsum

    # 리뷰 속도
    d2 = d.sort_values(["asin", "date"])
    rv = pd.to_numeric(d2["review_count"], errors="coerce")
    d2["rev_delta"] = rv.groupby(d2["asin"]).diff()
    d2.loc[d2["rev_delta"] < 0, "rev_delta"] = np.nan
    idx["review_velocity"] = d2.groupby("date")["rev_delta"].sum(min_count=1).rolling(28).sum()

    print("[지수] %d일 재계산 완료 (alpha=%.2f)" % (len(idx), ALPHA))
    print()
    print(idx.tail(8).round(4).to_string())
elif len(idx_db):
    idx = idx_db.set_index("date")[["gmv_index_all", "gmv_index_fixed",
                                    "gmv_index_dev", "brand_strength",
                                    "n_active_asin", "review_velocity"]]
    idx.columns = ["gmv_sum", "gmv_balanced", "gmv_device",
                   "brand_strength", "n_active", "review_velocity"]
    idx["gmv_mean"] = idx["gmv_sum"] / idx["n_active"].replace(0, np.nan)
    print("[지수] DB 저장분 사용 (%d행)" % len(idx))


In [ ]:
# ---------------------------------------------------------------- 3종 비교 국면 판정
if not idx.empty and len(idx) >= WINDOW_D * 2:
    rec = idx.iloc[-WINDOW_D:].mean(numeric_only=True)
    pri = idx.iloc[-WINDOW_D * 2:-WINDOW_D].mean(numeric_only=True)

    def _chg(k):
        a, b = rec.get(k), pri.get(k)
        if a is None or b is None or pd.isna(a) or pd.isna(b) or b == 0:
            return np.nan
        return a / b - 1

    print("=" * 78)
    print("국면 재판정 — 최근 %d일 vs 직전 %d일 (커버리지 보정 후)" % (WINDOW_D, WINDOW_D))
    print("=" * 78)
    rows = []
    for k, label, sens in [
        ("gmv_sum",      "gmv_sum      (합계)",   "높음"),
        ("gmv_mean",     "gmv_mean     (평균)",   "낮음"),
        ("gmv_balanced", "gmv_balanced (균형패널)", "없음"),
        ("gmv_device",   "gmv_device   (디바이스)", "중간"),
    ]:
        v = _chg(k)
        rows.append({"지수": label, "변화율": v, "커버리지민감도": sens})
        print("  %-26s %+8.1f%%   (민감도 %s)"
              % (label, v * 100 if pd.notna(v) else np.nan, sens))

    print("  %-26s %8.0f -> %.0f" % ("활성 ASIN 수",
                                     pri.get("n_active", np.nan), rec.get("n_active", np.nan)))
    print()

    s, m, b = _chg("gmv_sum"), _chg("gmv_mean"), _chg("gmv_balanced")
    if pd.notna(b):
        signs = [np.sign(x) for x in [s, m, b] if pd.notna(x)]
        if len(set(signs)) == 1:
            print("  [일관] 3종 지수 방향이 일치합니다 → 신호로 해석 가능")
            print("         변화율: %+.1f%% (균형패널 기준)" % (b * 100))
        else:
            print("  [불일치] 지수마다 방향이 다릅니다 → 커버리지 문제")
            print("           gmv_balanced(%+.1f%%)를 기준으로 판단하십시오." % (b * 100))
            if pd.notna(s) and pd.notna(b) and s < b:
                print("           gmv_sum이 더 나쁜 것은 패널에서 ASIN이 빠졌기 때문입니다.")
                print("           수요 감소가 아니라 데이터 커버리지 축소입니다.")
    print()
    print("  * Google Trends 지표와 교차 확인하십시오.")
else:
    print("[국면] 관측치 부족")


## 6. 시각화

### 6-1. 커버리지와 지수를 나란히

**이 그림이 이 노트북의 핵심이다.** 아래 패널의 활성 ASIN 수가 흔들리는 구간에서는
위 패널의 지수 변화를 수요로 읽으면 안 된다.

In [ ]:
if SHOW_PLOTS and not idx.empty:
    import matplotlib
    import matplotlib.pyplot as plt

    for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
        try:
            matplotlib.rcParams["font.family"] = _f
            break
        except Exception:
            continue
    matplotlib.rcParams["axes.unicode_minus"] = False

    base_dt = pd.to_datetime(REBASE_DATE) if REBASE_DATE else idx.index.min()

    def rebase(s: pd.Series) -> pd.Series:
        s = s.astype(float)
        anchor = s.loc[s.index >= base_dt].dropna()
        if anchor.empty or anchor.iloc[0] == 0:
            return s
        return s / anchor.iloc[0] * 100.0

    fig, axes = plt.subplots(2, 1, figsize=(FIG_W, 9), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})

    ax = axes[0]
    for col, lbl, c, lw in [
        ("gmv_sum",      "gmv_sum (합계, 커버리지 민감)", "#c0392b", 1.3),
        ("gmv_mean",     "gmv_mean (평균)",              "#f39c12", 1.3),
        ("gmv_balanced", "gmv_balanced (균형패널)",       "#2980b9", 2.2),
    ]:
        if col in idx.columns:
            ax.plot(idx.index, rebase(idx[col]), color=c, linewidth=lw, label=lbl, alpha=0.9)
    ax.axhline(100, color="grey", linestyle=":", linewidth=1)
    ax.set_ylabel("지수 (기준일=100)")
    ax.set_title("GMV proxy 지수 3종 비교 — 파란선(균형패널)이 가장 신뢰할 만함",
                 fontsize=12, loc="left")
    ax.legend(fontsize=9); ax.grid(alpha=0.25)

    ax = axes[1]
    ax.fill_between(idx.index, idx["n_active"], color="#7f8c8d", alpha=0.45,
                    step="post")
    ax.plot(idx.index, idx["n_active"], color="#2c3e50", linewidth=1.0,
            drawstyle="steps-post")
    ax.set_ylabel("활성 ASIN")
    ax.set_title("활성 ASIN 수 — 이 선이 흔들리는 구간의 gmv_sum 변화는 산술 효과",
                 fontsize=10, loc="left")
    ax.grid(alpha=0.25)

    plt.tight_layout(); plt.show()


### 6-2. ASIN별 데이터 가용 구간

가로 막대 하나가 ASIN 하나의 관측 구간이다. 오른쪽 끝이 들쭉날쭉하면
그것이 최근 커버리지 붕괴의 원인이다. 신규 SKU의 출시 시점도 여기서 읽힌다.

In [ ]:
if SHOW_PLOTS and len(daily):
    import matplotlib.pyplot as plt

    d = daily[daily["bsr_root"].notna()]
    span = d.groupby("asin")["date"].agg(["min", "max", "count"])
    tmap = master.set_index("asin")["title"].to_dict() if len(master) else {}
    dmap = master.set_index("asin")["is_device"].to_dict() if len(master) else {}
    span["title"] = [str(tmap.get(a, a))[:42] for a in span.index]
    span["is_device"] = [int(dmap.get(a, 0)) for a in span.index]
    span = span.sort_values("min")

    fig, ax = plt.subplots(figsize=(FIG_W, max(4, len(span) * 0.32)))
    for i, (asin, r) in enumerate(span.iterrows()):
        c = "#c0392b" if r["is_device"] else "#2980b9"
        ax.barh(i, (r["max"] - r["min"]).days, left=r["min"], height=0.6,
                color=c, alpha=0.75)
    ax.set_yticks(range(len(span)))
    ax.set_yticklabels(span["title"], fontsize=7)
    ax.invert_yaxis()
    if ANALYSIS_END is not None:
        ax.axvline(ANALYSIS_END, color="green", linestyle="--", linewidth=1.4,
                   label="분석 종료일")
    ax.axvline(daily["date"].max(), color="red", linestyle="--", linewidth=1.4,
               label="데이터 최종일")
    ax.set_title("ASIN별 데이터 가용 구간 — 파랑=스킨케어, 빨강=디바이스",
                 fontsize=11, loc="left")
    ax.legend(fontsize=9); ax.grid(alpha=0.2, axis="x")
    plt.tight_layout(); plt.show()


### 6-3. 개별 SKU BSR 추이

지수가 뭉개버린 개별 움직임을 본다. 로그 스케일이며 **아래로 갈수록 순위가 높다**
(y축을 뒤집어 위쪽이 상위가 되도록 그린다).

In [ ]:
if SHOW_PLOTS and len(daily):
    import matplotlib.pyplot as plt

    d = daily[daily["bsr_root"].notna()].copy()
    if "is_oos" in d.columns:
        d = d[d["is_oos"] == 0]
    rank = d.groupby("asin")["bsr_root"].median().nsmallest(TOP_N_SKU)
    tmap = master.set_index("asin")["title"].to_dict() if len(master) else {}

    n = len(rank)
    ncol = 3
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(FIG_W, nrow * 2.3), sharex=True)
    axes = np.atleast_1d(axes).ravel()

    for i, asin in enumerate(rank.index):
        ax = axes[i]
        sub = d[d["asin"] == asin].sort_values("date")
        ax.plot(sub["date"], sub["bsr_root"], linewidth=0.9, color="#2c3e50")
        ax.plot(sub["date"], sub["bsr_root"].rolling(28, min_periods=5).median(),
                linewidth=1.6, color="#c0392b")
        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_title(str(tmap.get(asin, asin))[:38], fontsize=7.5, loc="left")
        ax.grid(alpha=0.2)
        ax.tick_params(labelsize=7)

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle("개별 SKU BSR (로그, 위쪽=상위) — 빨간선은 28일 중앙값",
                 fontsize=11, y=1.002)
    plt.tight_layout(); plt.show()


### 6-4. SKU별 기여도

지수를 누가 끌고 가는지 본다. 소수 SKU에 집중돼 있으면 그 SKU의 재고·순위 변동이
그대로 지수 변동이 되므로, 신호의 견고성이 낮다는 뜻이다.

In [ ]:
if SHOW_PLOTS and len(daily) and ANALYSIS_START is not None:
    import matplotlib.pyplot as plt

    d = daily[(daily["date"] >= ANALYSIS_START) & (daily["date"] <= ANALYSIS_END)].copy()
    if "is_oos" in d.columns:
        d.loc[d["is_oos"] == 1, "bsr_root"] = np.nan
    bsr = pd.to_numeric(d["bsr_root"], errors="coerce").where(lambda x: x > 0)
    price = (pd.to_numeric(d["price_new"], errors="coerce")
             .fillna(pd.to_numeric(d.get("price_buybox"), errors="coerce")))
    d["gmv_proxy"] = (bsr ** (-ALPHA)) * price

    piv = d.pivot_table(index="date", columns="asin", values="gmv_proxy", aggfunc="mean")
    piv = piv.resample("MS").mean()
    top = piv.mean().nlargest(8).index.tolist()
    other = [c for c in piv.columns if c not in top]

    plot_df = piv[top].copy()
    if other:
        plot_df["기타 %d개" % len(other)] = piv[other].sum(axis=1)

    tmap = master.set_index("asin")["title"].to_dict() if len(master) else {}
    plot_df.columns = [str(tmap.get(c, c))[:30] if c in tmap else c for c in plot_df.columns]

    fig, axes = plt.subplots(2, 1, figsize=(FIG_W, 9))

    ax = axes[0]
    ax.stackplot(plot_df.index, plot_df.T.fillna(0).values,
                 labels=plot_df.columns, alpha=0.85)
    ax.set_title("월별 GMV proxy 기여도 (절대)", fontsize=11, loc="left")
    ax.legend(fontsize=6.5, ncol=2, loc="upper left")
    ax.grid(alpha=0.2)

    ax = axes[1]
    share = plot_df.div(plot_df.sum(axis=1), axis=0).fillna(0)
    ax.stackplot(share.index, share.T.values, labels=share.columns, alpha=0.85)
    ax.set_ylim(0, 1)
    ax.set_title("월별 기여도 비중 (상대) — 집중도가 높으면 신호가 취약",
                 fontsize=11, loc="left")
    ax.grid(alpha=0.2)

    plt.tight_layout(); plt.show()

    hhi = (share.iloc[-1] ** 2).sum()
    print("최근월 허핀달지수(HHI): %.3f" % hhi)
    print("  0.25 이상이면 소수 SKU 집중. 그 SKU의 품절·순위변동이 지수를 지배합니다.")


### 6-5. Google Trends 겹쳐보기

두 지표가 **어긋날 때 정보량이 가장 크다.**

- 검색 ↓ / BSR 유지 → Amazon 내부 검색·재구매로 전환된 성숙 국면일 수 있음
- 검색 ↑ / BSR ↓ → 관심은 있으나 전환이 안 됨. 재고 문제 또는 경쟁 심화

In [ ]:
if SHOW_PLOTS and OVERLAY_TRENDS and DB_AVAILABLE and not idx.empty:
    try:
        tq = ("SELECT date, brand_level, sos, collected_at FROM {t} "
              "WHERE geo='US' ORDER BY date").format(t=TBL_TRENDS_MET)
        tr = pd.read_sql(text(tq), ENGINE)
    except Exception as e:
        tr = pd.DataFrame()
        print("[Trends] 조회 실패 (아직 수집 전일 수 있음):", repr(e)[:120])

    if len(tr):
        tr["date"] = pd.to_datetime(tr["date"])
        tr = tr[tr["collected_at"] == tr["collected_at"].max()]
        tr = tr.set_index("date").sort_index()

        import matplotlib.pyplot as plt
        base_dt = pd.to_datetime(REBASE_DATE) if REBASE_DATE else idx.index.min()

        def rb(s):
            s = s.astype(float).dropna()
            a = s.loc[s.index >= base_dt]
            return s / a.iloc[0] * 100 if len(a) and a.iloc[0] else s

        fig, ax = plt.subplots(figsize=(FIG_W, 5))
        col = "gmv_balanced" if "gmv_balanced" in idx.columns else "gmv_sum"
        ax.plot(idx.index, rb(idx[col]).rolling(28, min_periods=7).mean(),
                color="#2980b9", linewidth=2.0, label="Amazon %s (28일 평균)" % col)
        ax.plot(tr.index, rb(tr["brand_level"]).rolling(4, min_periods=2).mean(),
                color="#c0392b", linewidth=2.0, label="Trends brand_level (4주 평균)")
        ax.axhline(100, color="grey", linestyle=":", linewidth=1)
        ax.set_ylabel("지수 (기준일=100)")
        ax.set_title("Amazon 지수 vs Google Trends — 어긋나는 구간을 주목",
                     fontsize=11, loc="left")
        ax.legend(fontsize=9); ax.grid(alpha=0.25)
        plt.tight_layout(); plt.show()
    else:
        print("[Trends] 데이터 없음 — apr_us_google_trends_v2 을 먼저 실행하십시오.")


## 7. 진단 요약

In [ ]:
print("=" * 78)
print("진단 요약")
print("=" * 78)
if len(daily):
    print("  ASIN           : %d개" % daily["asin"].nunique())
    print("  원본 기간      : %s ~ %s"
          % (daily["date"].min().date(), daily["date"].max().date()))
if ANALYSIS_START is not None:
    print("  분석 기간      : %s ~ %s" % (ANALYSIS_START.date(), ANALYSIS_END.date()))
    print("  꼬리 절단      : %d일" % (daily["date"].max() - ANALYSIS_END).days)
if not idx.empty and len(idx) >= WINDOW_D * 2:
    r = idx.iloc[-WINDOW_D:].mean(numeric_only=True)
    p = idx.iloc[-WINDOW_D * 2:-WINDOW_D].mean(numeric_only=True)
    for k in ["gmv_sum", "gmv_mean", "gmv_balanced"]:
        if k in r and p.get(k):
            print("  %-14s : %+.1f%%" % (k, (r[k] / p[k] - 1) * 100))
print()
print("  [해석 원칙]")
print("   1. gmv_balanced 가 기준입니다. 구성 변화의 영향을 받지 않습니다.")
print("   2. gmv_sum 만 나쁘면 그것은 커버리지 축소이지 수요 감소가 아닙니다.")
print("   3. 3종이 같은 방향일 때만 신호로 읽으십시오.")
print("   4. 최종 판단 전 Google Trends 와 교차 확인하십시오.")
print("=" * 78)


---

## 커버리지가 계속 무너진다면

수집 노트북(`apr_us_amazon_keepa_v2`)에서 다음을 조정한다.

| 증상 | 조정 |
|---|---|
| 꼬리 절단이 30일 이상 | `FFILL_LIMIT_D` 를 7 → 14로 상향 |
| 특정 ASIN만 계속 빠짐 | 단종·비활성 상품. 마스터에서 `is_active=0` 처리 |
| 전반적으로 관측이 희소 | Keepa가 저순위 상품을 드물게 갱신하는 것. `MIN_HISTORY_D` 상향 |

`FFILL_LIMIT_D`를 늘리면 커버리지는 좋아지지만 보간 비율이 올라간다.
`is_ffilled` 비율이 60%를 넘어가면 그 지수는 관측이 아니라 추정에 가깝다.

## 다음 단계

1. 3종 지수 방향이 일치하는 구간만 골라 APR 분기 매출과 상관 재검증
2. Trends 지표와의 괴리 구간을 별도로 표시해 이벤트(품절·프로모션·바이럴)와 대조
3. 다종목 확장 시 `TICKER`만 바꿔 같은 노트북 재사용

---
*V2 · 2026-08 · 짝 노트북: `apr_us_amazon_keepa_v2`, `apr_us_google_trends_v2`*